# Analysis (spec section 13)

Reads a run's JSONL logs and cost ledger and draws the headline figures. **Scope: the logs come from the phase-2 dry run on the stub backend** (3 groups of 4 agents, 200 generations, in-group teaching with self-play verification). Agents are perturbed copies of anchor rule lists, so curves show that the pipeline measures what it should, not that anything learns. Token and dollar axes are the stub's fake usage priced at Haiku 4.5 list prices.

In [1]:
import json, os, sys
sys.path.insert(0, 'src')
%matplotlib inline
from culture.analysis.metrics import load_run, series, trajectory_shape, adoption_edges, retained_innovations
from culture.analysis import figures
from culture.analysis.report import run_report
RUN = os.environ.get('ANALYSIS_RUN', 'runs/dry/ref')
run = load_run(RUN)
s = series(run)
print(RUN, len(run['generations']), 'generations')

runs/dry/ref 200 generations


## Headline: between-group cross-play versus cumulative tokens

Dashed lines are external reference points from the literature (OBL cross-play, humans, SmartBot, o3 with deductions). In the stub dry run the between-group number sits near self-play because every stub bot shares one template.

In [2]:
figures.between_vs_tokens(s);

In [3]:
figures.score_vs_generation(s);

In [4]:
figures.teaching_rates(s);

In [5]:
figures.tokens_by_tag(s);

In [6]:
figures.diversity(s);

## Teaching and adoption graph

Adoption edges (sender -> receiver) within and across groups. The DCMM community fit (phase 5) is added below once built.

In [7]:
edges = adoption_edges(run)
within = sum(e['sender_group'] == e['receiver_group'] for e in edges)
print(f'{len(edges)} adoptions: {within} within groups, {len(edges) - within} across groups')
print('retained innovations (k=10):', retained_innovations(run))
print('trajectory shape:', trajectory_shape(s, threshold=17.0))

567 adoptions: 567 within groups, 0 across groups
retained innovations (k=10): 7
trajectory shape: {'slope_first_third': 0.0399382023449191, 'slope_last_third': -0.0027943325331383825, 'changepoints': [5, 29], 'first_gen_between_above_threshold': 9}


## Run report

In [8]:
from IPython.display import Markdown
Markdown(run_report(RUN))

# Run dryrun / default (population seed 0)

Backend: stub (stub numbers are not model evidence). Config digest 1ed0570db0f30055. Generations: 200.

| metric | first | last |
|---|---|---|
| population mean self-play | 9.04 | 18.10 |
| population best self-play | 17.15 | 18.30 |
| between-group cross-play | 16.62 | 17.58 |
| distinct incumbents | 12 | 11 |
| cumulative tokens | 90523 | 30442184 |
| cumulative spend (USD, nominal for stub) | 0.21 | 50.28 |

Slope of population mean: first third +0.0399/gen, last third -0.0028/gen; changepoints at generations [5, 29].
Adoptions: 567 (0 across groups). Messages delivered: 7164; verification pass rate overall 0.09.
Retained innovations (k=10): 7.
Credit (cumulative) by agent: g0a0 6.12, g0a1 16.05, g0a2 -0.48, g0a3 -3.89, g1a0 18.45, g1a1 1.19, g1a2 22.50, g1a3 1.20, g2a0 8.98, g2a1 10.34, g2a2 3.27, g2a3 -1.34


## Resume check and log growth (phase 2)

In [9]:
p = 'runs/dry/kill_resume_check.json'
print(open(p).read() if os.path.exists(p) else 'not run')

{
 "config": "configs/dryrun.yaml",
 "killed_with": "SIGKILL to the process group",
 "checkpoint_generation_at_kill": 100,
 "log_bytes_at_kill": {
  "generations": 1247510,
  "ledger": 1058126,
  "messages": 4174403,
  "artifacts": 298092
 },
 "ref_returncode": 0,
 "resume_returncode": 0,
 "identical": true,
 "digest_ref": {
  "state": "e04f3b1b33de39f9",
  "generations": "831eb56a4b1ba9d6",
  "artifacts": "feecb9a046d20fba",
  "messages": "714dd898999194c7",
  "ledger": "cb50546487d57f0d",
  "touch": "bcb1c864091adc3a",
  "provenance": "a1b3293647212334",
  "artifact_files": "ef118315038c2d52"
 },
 "digest_resumed": {
  "state": "e04f3b1b33de39f9",
  "generations": "831eb56a4b1ba9d6",
  "artifacts": "feecb9a046d20fba",
  "messages": "714dd898999194c7",
  "ledger": "cb50546487d57f0d",
  "touch": "bcb1c864091adc3a",
  "provenance": "a1b3293647212334",
  "artifact_files": "ef118315038c2d52"
 },
 "generations": 200,
 "generation_record_bytes": {
  "first10_mean": 11931.2,
  "last10_mean":

## Section-10 toolkit on the dry-run logs (phase 5)

Each tool is validated on synthetic data with a known answer in `tests/test_analysis_phase5.py`; here they run on the stub dry run, where the expected answers are mostly structural (isolated groups should come back as separate idea-communities).

### Higher Criticism with Efron's empirical null

Per generation: each agent's paired z-score of its incumbent against the previous generation's on the same deals. With 12 agents the test has little power; the point is the plumbing and the plateau signal.

In [10]:
import numpy as np
from culture.analysis.hc import hc_test, plateau
res = [hc_test(r['hc']['z'], null='empirical', sims=300) if len(r['hc']['z']) >= 5 else None for r in run['generations']]
det = [r.detected for r in res if r is not None]
print(f'generations with HC detection: {sum(det)} of {len(det)}; first plateau (5 generations under threshold) starts at generation {plateau(res, 5)}')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.5, 3))
g = [r['generation'] for r, x in zip(run['generations'], res) if x is not None]
ax.plot(g, [x.hc for x in res if x is not None], color=figures.SERIES[0], linewidth=2, label='HC (empirical null)')
ax.plot(g, [x.threshold for x in res if x is not None], color=figures.MUTED, linewidth=1, linestyle=(0, (3, 3)), label='null 95% threshold')
figures._style(ax, 'Did anyone improve? Higher Criticism per generation', 'generation', 'HC+')
ax.legend(frameon=False, fontsize=8);

generations with HC detection: 23 of 199; first plateau (5 generations under threshold) starts at generation 1


### DCMM on the adoption graph

In [11]:
from culture.analysis.graphs import membership_over_time, mixed_score, adjacency, nmi, influence
nodes = sorted(run['generations'][0]['agents'])
groups = [run['generations'][-1]['agents'][a]['group'] for a in nodes]
edges = adoption_edges(run)
A = adjacency(edges, nodes)
fit = mixed_score(A + A.T, K=len(set(groups)), nodes=nodes)
print('NMI(recovered communities, configured groups) =', round(nmi(fit.hard(), groups), 3))
print('cross-lineage leakage (mean off-diagonal of P) =', round(fit.leakage(), 3))
inf = influence(edges, nodes)
print('top influence (adoptions caused):', sorted(inf.items(), key=lambda kv: -kv[1])[:4])
print('theta (degree parameter):', dict(zip(nodes, np.round(fit.theta, 2))))
track = membership_over_time(edges, nodes, len(set(groups)), list(range(20, len(run['generations']), 20)), window=40)
print('NMI over time:', [(t['generation'], round(nmi(t['pi'].argmax(1), groups), 2)) for t in track if t['pi'] is not None])

NMI(recovered communities, configured groups) = 1.0
cross-lineage leakage (mean off-diagonal of P) = 0.064
top influence (adoptions caused): [('g0a1', 77.0), ('g1a0', 73.0), ('g0a0', 63.0), ('g2a0', 62.0)]
theta (degree parameter): {'g0a0': np.float64(1.0), 'g0a1': np.float64(0.97), 'g0a2': np.float64(0.82), 'g0a3': np.float64(0.75), 'g1a0': np.float64(0.94), 'g1a1': np.float64(0.83), 'g1a2': np.float64(0.68), 'g1a3': np.float64(0.73), 'g2a0': np.float64(0.92), 'g2a1': np.float64(0.84), 'g2a2': np.float64(0.77), 'g2a3': np.float64(0.77)}
NMI over time: [(20, np.float64(0.67)), (40, np.float64(1.0)), (60, np.float64(1.0)), (80, np.float64(1.0)), (100, np.float64(1.0)), (120, np.float64(1.0)), (140, np.float64(1.0)), (160, np.float64(0.67)), (180, np.float64(1.0))]


### IF-PCA: how many distinct strategies?

Behavioural fingerprints of the distinct incumbents of the last 50 generations on a fixed probe set of 200 observations (one-hot action category per probe). On binary features the KS screen keeps every feature on which artifacts disagree, so this reduces to spectral clustering on those.

In [12]:
from culture.analysis.behavior import probe_set, actions, features, disagreement
from culture.analysis.ifpca import ifpca
from culture.artifacts.store import ArtifactStore
store = ArtifactStore(RUN + '/artifacts')
ids = sorted({a['incumbent'] for r in run['generations'][-50:] for a in r['agents'].values()})
probes = probe_set(200)
acts = [actions(store.get(i).code, probes) for i in ids]
X = features(acts)
r = ifpca(X) if len(ids) >= 6 else None
D = disagreement(acts)
print(f'{len(ids)} distinct incumbents; IF-PCA strategies K = {r.K if r else None}; features kept {len(r.selected) if r else 0} of {X.shape[1]}')
print('mean pairwise action disagreement on the probe set:', round(float(D[np.triu_indices(len(ids), 1)].mean()), 3))

295 distinct incumbents; IF-PCA strategies K = 6; features kept 54 of 800
mean pairwise action disagreement on the probe set: 0.115
